In [1]:
import pandas as pd
import numpy as np

data_df = pd.read_csv("../data/cleaned_traffic.csv")
data_df.head()

,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,...,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity,Accident_severity_encoded
0,17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,Above 10yr,...,Going straight,na,na,na,Unknown,Unknown,Not a Pedestrian,Moving Backward,Slight Injury,2
1,17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,5-10yrs,...,Going straight,na,na,na,Unknown,Unknown,Not a Pedestrian,Overtaking,Slight Injury,2
2,17:02:00,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,Unknown,...,Going straight,Driver or rider,Male,31-50,Driver,Unknown,Not a Pedestrian,Changing lane to the left,Serious Injury,1
3,1:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,Unknown,...,Going straight,Pedestrian,Female,18-30,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury,2
4,1:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,Unknown,Owner,5-10yrs,...,Going straight,na,na,na,Unknown,Unknown,Not a Pedestrian,Overtaking,Slight Injury,2


In [2]:
data_df["Hour"] = pd.to_datetime(
    data_df["Time"],
    format= "%H:%M:%S"
).dt.hour # this only access hour from time column

In [3]:
data_df[["Time", "Hour"]].head(10)

,Time,Hour
0,17:02:00,17
1,17:02:00,17
2,17:02:00,17
3,1:06:00,1
4,1:06:00,1
5,14:15:00,14
6,17:30:00,17
7,17:20:00,17
8,17:20:00,17
9,17:20:00,17


The original Time column contains hours, minutes, and seconds. By extracting the hour, I created a new numerical feature called Hour.


# group the hours into different periods of the day

In [4]:
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

In [5]:
data_df["Time_of_day"] = data_df["Hour"].map(get_time_of_day)

In [6]:
data_df[["Time", "Hour", "Time_of_day"]].head(10)

,Time,Hour,Time_of_day
0,17:02:00,17,Evening
1,17:02:00,17,Evening
2,17:02:00,17,Evening
3,1:06:00,1,Night
4,1:06:00,1,Night
5,14:15:00,14,Afternoon
6,17:30:00,17,Evening
7,17:20:00,17,Evening
8,17:20:00,17,Evening
9,17:20:00,17,Evening


I created a new feature called Time_of_day by grouping accident hours into Morning, Afternoon, Evening, and Night.
The purpose of creating this feature is to help identify whether accident severity is associated with different times of the day.


# Weekend Feature

In [7]:
data_df["Is_weekend"] = data_df["Day_of_week"].isin(
    ["Saturday", "Sunday"]
).astype(int)

In [8]:
data_df[["Day_of_week", "Is_weekend"]].head(10)

,Day_of_week,Is_weekend
0,Monday,0
1,Monday,0
2,Monday,0
3,Sunday,1
4,Sunday,1
5,Friday,0
6,Wednesday,0
7,Friday,0
8,Friday,0
9,Friday,0


The purpose of this feature is to help identify whether accident severity is associated with weekends or weekdays. where 0 = not weekend and 1 = weekend

In [9]:
data_df["Time_of_day"].value_counts() #distribution accross the day, night

Time_of_day
Afternoon    2582
Evening      2280
Morning      2223
Night        1125
Name: count, dtype: int64

In [10]:
# reelationship between tiime of day and acccident severity
pd.crosstab(
    data_df["Time_of_day"], data_df["Accident_severity"]
    
)

Accident_severity,Fatal injury,Serious Injury,Slight Injury
Time_of_day,,,
Afternoon,14,319,2249
Evening,28,284,1968
Morning,16,273,1934
Night,24,170,931


The results showed that Afternoon had the highest number of slight injury records (2,249), while Evening had the highest number of fatal injury records (28) Although Night had the fewest accident records, it had 24 fatal injury records.

In [11]:
pd.crosstab(
    data_df["Time_of_day"],
    data_df["Accident_severity"],
    normalize="index"
) * 100

Accident_severity,Fatal injury,Serious Injury,Slight Injury
Time_of_day,,,
Afternoon,0.542215,12.354764,87.103021
Evening,1.228070,12.456140,86.315789
Morning,0.719748,12.280702,86.999550
Night,2.133333,15.111111,82.755556


In [12]:
# weekend feature against accident severity
pd.crosstab(
    data_df["Is_weekend"],
    data_df["Accident_severity"],
    normalize="index"
) * 100

Accident_severity,Fatal injury,Serious Injury,Slight Injury
Is_weekend,,,
0,0.680493,12.491899,86.827609
1,1.962709,13.493621,84.543670


In [13]:
data_df[
    ["Time", "Hour", "Time_of_day", "Day_of_week", "Is_weekend"]
    ].head(10)

,Time,Hour,Time_of_day,Day_of_week,Is_weekend
0,17:02:00,17,Evening,Monday,0
1,17:02:00,17,Evening,Monday,0
2,17:02:00,17,Evening,Monday,0
3,1:06:00,1,Night,Sunday,1
4,1:06:00,1,Night,Sunday,1
5,14:15:00,14,Afternoon,Friday,0
6,17:30:00,17,Evening,Wednesday,0
7,17:20:00,17,Evening,Friday,0
8,17:20:00,17,Evening,Friday,0
9,17:20:00,17,Evening,Friday,0


In [14]:
# check if newly created columns are null
data_df[["Hour", "Time_of_day", "Is_weekend"]].isnull().sum()

Hour           0
Time_of_day    0
Is_weekend     0
dtype: int64

In [15]:
data_df.shape

(8210, 35)

In [17]:
# removing column time that is not use now
data_df = data_df.drop(columns=["Time"])

In [18]:
data_df.shape

(8210, 34)

This is the final dataset

# saving model

In [19]:
data_df.to_csv(
    "../data/feature_engineered_traffic.csv", index=False

)